Importing all essential libraries


Here I have imported all necessary libraries for data handling, text processing, and machine learning.

In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import warnings
warnings.filterwarnings('ignore')

Downloading required NLTK resources


Here I have downloaded the necessary NLTK packages for text preprocessing.

In [64]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gchaw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gchaw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [65]:
def load_dataset():
    file_path = r"C:\Users\gchaw\OneDrive\Desktop\pythonwork\Ai assignment Vijay wfh\IMDB Dataset.csv"
    df = pd.read_csv(file_path)  # Use absolute path
    df.dropna(inplace=True)  # Remove any missing values
    return df

In [66]:
df = load_dataset()  # Call the function and store the dataset
df.head()  # Now it should work

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Here I have initialized the TF-IDF vectorizer to convert text into numerical form.


TF-IDF helps in giving importance to meaningful words while reducing noise.

In [67]:
def preprocess_review(text):
    """
    Clean and preprocess the review text.
    """
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and digits
    
    tokens = word_tokenize(text)  # Tokenize text
    stop_words = set(stopwords.words('english'))  # Get English stopwords
    tokens = [token for token in tokens if token not in stop_words]  # Remove stopwords
    
    return ' '.join(tokens)  # Return cleaned text

In [69]:
df['cleaned_review'] = df['review'].apply(preprocess_review)  # Apply cleaning
df.head()  # Check cleaned data

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [70]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)  # Initialize TF-IDF Vectorizer

In [71]:
X = tfidf_vectorizer.fit_transform(df['cleaned_review'])  # Convert text to numerical features
y = df['sentiment'].map({'positive': 1, 'negative': 0})  # Convert labels to binary (1=positive, 0=negative)


In [74]:
from sklearn.model_selection import train_test_split

# Splitting dataset into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display shapes of training and testing sets
print("Training Data Shape:", X_train.shape, y_train.shape)
print("Testing Data Shape:", X_test.shape, y_test.shape)


Training Data Shape: (40000, 5000) (40000,)
Testing Data Shape: (10000, 5000) (10000,)


Here I have chosen Logistic Regression as the model for sentiment classification.


Logistic Regression is a simple yet powerful classifier for text classification.

In [75]:
from sklearn.linear_model import LogisticRegression

# Initialize and train Logistic Regression model
model = LogisticRegression(max_iter=1000)  # Set max iterations to 1000 for better convergence
model.fit(X_train, y_train)  # Train the model

print("Model training complete.")

Model training complete.


In [76]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict sentiments for test set
y_pred = model.predict(X_test)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy*100:.2f}%")
print(f"Model Precision: {precision*100:.2f}%")
print(f"Model Recall: {recall*100:.2f}%")

Model Accuracy: 88.67%
Model Precision: 87.75%
Model Recall: 90.10%


In [77]:
def predict_sentiment(review_text, model, tfidf_vectorizer):
    # Preprocess the review
    processed_review = preprocess_review(review_text)

    # Convert to TF-IDF features
    review_tfidf = tfidf_vectorizer.transform([processed_review])

    # Predict sentiment
    prediction = model.predict(review_tfidf)

    return "Positive" if prediction[0] == 1 else "Negative"

Here I am testing the trained model on sample reviews to check if it predicts correctly.

In [78]:
sample_reviews = [
    "This movie was absolutely fantastic! The acting was superb.",
    "I really hated this film. It was boring and a waste of time.",
    "It was an okay movie, nothing too special."
]

for review in sample_reviews:
    sentiment = predict_sentiment(review, model, tfidf_vectorizer)
    print(f"Review: {review}\nPredicted Sentiment: {sentiment}\n")


Review: This movie was absolutely fantastic! The acting was superb.
Predicted Sentiment: Positive

Review: I really hated this film. It was boring and a waste of time.
Predicted Sentiment: Negative

Review: It was an okay movie, nothing too special.
Predicted Sentiment: Negative

